# TP5 — Multimodality and Generative AI

## Objectives
- Set up audio pipelines: Text-to-Speech and Speech-to-Text.
- Analyse images with vision models.
- Generate images from text prompts.
- Compose a full multimodal pipeline: image → text → audio.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.getenv('OPENAI_API_KEY'), 'Missing OPENAI_API_KEY'
print('OpenAI key loaded.')


## Step 2 — Text-to-Speech (Ex 2.1, 2.2)


In [ ]:
import time
from pathlib import Path
from openai import OpenAI

client = OpenAI()
OUTPUT_AUDIO = Path('outputs/audio')
OUTPUT_AUDIO.mkdir(parents=True, exist_ok=True)

def synthesize_message(message, out_path, voice='alloy', language='en-US'):
    output = Path(out_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    instructions = f'Speak in {language} with clear and natural pronunciation.'
    t0 = time.perf_counter()
    with client.audio.speech.with_streaming_response.create(
        model='gpt-4o-mini-tts', voice=voice, input=message, instructions=instructions
    ) as r:
        r.stream_to_file(output)
    print(f'[{voice}] {time.perf_counter()-t0:.2f}s \u2192 {output}')
    return output

TEXT = 'Hello! Welcome to the multimodality lab. Today we explore text-to-speech and beyond.'

# Ex 2.1: compare three voices
for voice in ['alloy', 'verse', 'nova', 'echo', 'fable']:
    synthesize_message(TEXT, OUTPUT_AUDIO / f'voice_{voice}.mp3', voice=voice)

# Ex 2.2: language variation
synthesize_message(
    'Bonjour ! Bienvenue dans le laboratoire de multimodalit\u00e9.',
    OUTPUT_AUDIO / 'french_intro.mp3', voice='verse', language='fr-FR'
)
print('Listen to outputs/audio/ to compare voices.')


## Step 3 — Speech-to-Text (Ex 3.1, 3.2)

> Place an MP3 file at `inputs/audio/meeting.mp3` to run this step.


In [ ]:
import json
import tempfile
import time
from pathlib import Path
from openai import OpenAI

try:
    from pydub import AudioSegment
    PYDUB_OK = True
except ImportError:
    PYDUB_OK = False
    print('pydub not installed — run: pip install pydub')

client = OpenAI()
AUDIO_FILE = Path('inputs/audio/meeting.mp3')
OUT_DIR    = Path('outputs/transcriptions')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def transcribe_mp3(mp3_path, chunk_minutes=2):
    audio    = AudioSegment.from_file(str(mp3_path))
    chunk_ms = chunk_minutes * 60 * 1000
    results  = []
    with tempfile.TemporaryDirectory() as wd:
        for i, start in enumerate(range(0, len(audio), chunk_ms)):
            seg = audio[start:start+chunk_ms]
            if len(seg) < 1000: continue
            cp = Path(wd) / f'chunk_{i}.mp3'
            seg.export(cp, format='mp3')
            t0 = time.perf_counter()  # Ex 3.1
            with cp.open('rb') as f:
                r = client.audio.transcriptions.create(
                    model='gpt-4o-mini-transcribe', file=f, response_format='verbose_json')
            elapsed = time.perf_counter() - t0
            results.append({'i': i, 'start_ms': start, 'end_ms': start+len(seg),
                            'text': r.text.strip(), 'api_time': round(elapsed,3)})
            print(f'Chunk {i}: {elapsed:.2f}s \u2192 "{r.text[:60]}..."')
    return results

if PYDUB_OK and AUDIO_FILE.exists():
    chunks = transcribe_mp3(AUDIO_FILE)
    full_text = ' '.join(c['text'] for c in chunks)
    (OUT_DIR / 'meeting.txt').write_text(full_text, encoding='utf-8')
    # Ex 3.2: save timeline JSON
    (OUT_DIR / 'meeting_timeline.json').write_text(
        json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Transcript and timeline saved.')
else:
    print('Place meeting.mp3 in inputs/audio/ to run transcription.')


## Step 4 — Vision: Image Understanding (Ex 4.1, 4.2)

> Place PNG/JPG images in `inputs/images/` to run this step.


In [ ]:
import base64
import json
import mimetypes
from pathlib import Path
from openai import OpenAI

client    = OpenAI()
IMG_DIR   = Path('inputs/images')
IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR   = Path('outputs/vision')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def encode_image(path):
    path = Path(path)
    mime, _ = mimetypes.guess_type(path.name)
    b64 = base64.b64encode(path.read_bytes()).decode()
    return f'data:{mime};base64,{b64}'

def describe_image(image_path, question, model='gpt-4.1-mini',
                   max_tokens=400, output_format='text'):
    format_hint = {
        'text':   '',
        'bullet': ' Answer as a bullet-point list (max 6 bullets).',
        'json':   ' Respond ONLY with JSON: {"summary":str,"key_points":[str],"concerns":[str]}',
    }.get(output_format, '')
    resp = client.responses.create(
        model=model,
        input=[{'role':'user','content':[
            {'type':'input_text',  'text': question+format_hint},
            {'type':'input_image', 'image_url': encode_image(image_path)},
        ]}],
        temperature=0.1, max_output_tokens=max_tokens,
    )
    return '\n'.join(
        c.text.strip() for item in resp.output
        for c in item.content if c.type=='output_text'
    )

images = list(IMG_DIR.glob('*.png')) + list(IMG_DIR.glob('*.jpg'))
if not images:
    print('No images found — place PNG/JPG files in inputs/images/')
else:
    img = images[0]
    Q   = 'What can you deduce from this image? Describe the main elements.'

    # Ex 4.1: compare models
    for model in ['gpt-4.1-mini', 'gpt-4.1']:
        ans = describe_image(img, Q, model=model)
        print(f'[{model}]\n{ans}\n')

    # Ex 4.2: format constraints
    for fmt in ['text', 'bullet', 'json']:
        ans = describe_image(img, Q, output_format=fmt)
        print(f'[{fmt}]\n{ans}\n')
        if fmt == 'json':
            try: json.loads(ans); print('JSON valid.')
            except: print('JSON parse failed — model did not comply.')


## Step 5 — Image Generation (Ex 5.1, 5.2)


In [ ]:
import base64
import time
from pathlib import Path
from openai import OpenAI

client  = OpenAI()
OUT_IMG = Path('outputs/images')
OUT_IMG.mkdir(parents=True, exist_ok=True)

def generate_image(prompt, out_path, size='1024x1024', quality='high'):
    t0  = time.perf_counter()
    res = client.images.generate(model='gpt-image-1', prompt=prompt,
                                  size=size, quality=quality)
    out = Path(out_path)
    out.write_bytes(base64.b64decode(res.data[0].b64_json))
    print(f'{out.name}: {time.perf_counter()-t0:.2f}s ({size}, q={quality})')
    return out

# Ex 5.1: size and quality comparison
for size, quality, label in [
    ('1024x1024','standard','square_standard'),
    ('1024x1024','high',    'square_high'),
    ('1792x1024','standard','landscape_standard'),
]:
    generate_image('Minimalist AI poster with sound wave and eye icons.',
                   OUT_IMG/f'{label}.png', size=size, quality=quality)

# Ex 5.2: three prompts for peer voting
prompts = [
    ('vote_1', 'Abstract neural network art: glowing nodes on dark background, neon colors.'),
    ('vote_2', 'Mountain lake at sunrise reflecting aurora borealis, photorealistic.'),
    ('vote_3', 'Retro-futuristic robot reading in a 1970s living room, pastel illustration.'),
]
for label, p in prompts:
    generate_image(p, OUT_IMG/f'{label}.png', size='1024x1024', quality='standard')
print('Vote with your partner: which image best matches its prompt?')


## Step 6 — Full Multimodal Pipeline (Ex 6.1, 6.2)


In [ ]:
import base64
import mimetypes
from pathlib import Path
from openai import OpenAI

client = OpenAI()

def encode_image(path):
    path = Path(path)
    mime, _ = mimetypes.guess_type(path.name)
    return f'data:{mime};base64,{base64.b64encode(path.read_bytes()).decode()}'

def describe_image(image_path, question, model='gpt-4.1-mini', max_tokens=400):
    resp = client.responses.create(
        model=model,
        input=[{'role':'user','content':[
            {'type':'input_text',  'text': question},
            {'type':'input_image', 'image_url': encode_image(image_path)},
        ]}],
        temperature=0.1, max_output_tokens=max_tokens,
    )
    return '\n'.join(c.text.strip() for item in resp.output for c in item.content if c.type=='output_text')

# Ex 6.1: translation before TTS
def translate_text(text, target='French'):
    resp = client.responses.create(
        model='gpt-4.1-mini',
        input=[{'role':'system','content':f'Translate to {target}. Return ONLY the translation.'},
               {'role':'user',  'content': text}],
        temperature=0.2, max_output_tokens=600,
    )
    return '\n'.join(c.text.strip() for item in resp.output for c in item.content if c.type=='output_text')

def synthesize(message, out_path, voice='verse', language='en-US'):
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with client.audio.speech.with_streaming_response.create(
        model='gpt-4o-mini-tts', voice=voice, input=message,
        instructions=f'Speak in {language} clearly.'
    ) as r:
        r.stream_to_file(out)
    return out

def visual_to_audio(image_path, question, voice='verse',
                    translate_to=None, out_audio=None):
    image_path = Path(image_path)
    if out_audio is None:
        out_audio = Path('outputs/audio') / f'analysis_{image_path.stem}.mp3'

    print('Step 1: Vision analysis...')
    summary = describe_image(image_path, question)
    print('Summary:', summary[:200])

    tts_text, lang = summary, 'en-US'
    if translate_to:  # Ex 6.1
        print(f'Ex 6.1: Translating to {translate_to}...')
        tts_text = translate_text(summary, target=translate_to)
        lang_map = {'french':'fr-FR','spanish':'es-ES','german':'de-DE'}
        lang = lang_map.get(translate_to.lower(), 'en-US')
        print('Translated:', tts_text[:200])

    print('Step 3: TTS...')
    audio_path = synthesize(f'Here is my analysis: {tts_text}',
                             out_audio, voice=voice, language=lang)
    print('Audio saved to:', audio_path)
    return audio_path

# Demo — requires an image in inputs/images/
images = list(Path('inputs/images').glob('*.png')) + list(Path('inputs/images').glob('*.jpg'))
if images:
    visual_to_audio(
        images[0],
        question='Identify the main elements and their significance.',
        voice='verse',
        translate_to='French',   # Ex 6.1: set to None for English-only
    )
else:
    print('Place images in inputs/images/ to run the pipeline demo.')

# Ex 6.2: CLI usage example (run from terminal)
print('\nEx 6.2 — CLI usage:')
print('  python tp5_pipeline.py inputs/images/photo.png --translate French --voice verse')
